# 사일로봇 결함탐지 — YOLOv8s-seg (crack / corrosion) 재학습

2클래스 인스턴스 세그멘테이션. 학습은 `/content/` 로컬, Drive는 데이터 zip·체크포인트 영속화용.

**사전 준비**
- Drive `/MyDrive/YOLOv8s/crack_corrosion_seg.zip` (로컬 `scripts/merge_and_split.py` 결과를 zip으로 업로드)
- 런타임: GPU (T4)


In [ ]:
# 1. 환경
!nvidia-smi -L
!pip -q install "ultralytics==8.4.56" onnx onnxslim
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# 2. Drive 마운트 + 데이터 준비 (/content 로 압축 해제)
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, pathlib
DRIVE = '/content/drive/MyDrive/YOLOv8s'
ZIP   = f'{DRIVE}/crack_corrosion_seg.zip'
DEST  = '/content/dataset'

assert os.path.exists(ZIP), f'zip 없음: {ZIP}'
pathlib.Path(DEST).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DEST)

# data.yaml 위치 자동 탐색 후 path 를 /content 기준으로 재작성
import glob, yaml
dy = glob.glob(f'{DEST}/**/data.yaml', recursive=True)[0]
root = os.path.dirname(dy)
d = yaml.safe_load(open(dy))
d['path'] = root
yaml.safe_dump(d, open(dy, 'w'), allow_unicode=True, sort_keys=False)
DATA = dy
print('DATA =', DATA); print(open(dy).read())


In [ ]:
# 3. 데이터 점검 (split x class 인스턴스 수)
import glob, collections, os
base = os.path.dirname(DATA)
for sp in ['train','val','test']:
    cnt = collections.Counter(); imgs = 0
    for t in glob.glob(f'{base}/labels/{sp}/*.txt'):
        imgs += 1
        for ln in open(t):
            p = ln.split()
            if p: cnt[int(float(p[0]))] += 1
    print(f'{sp:5s}  images={imgs:5d}  crack={cnt[0]:5d}  corrosion={cnt[1]:5d}')


In [ ]:
# 4. 학습
from ultralytics import YOLO
RUN_PROJECT = f'{DRIVE}/silorobot_runs'
RUN_NAME    = 'cc_seg_v1'

model = YOLO('yolov8s-seg.pt')
results = model.train(
    data=DATA,
    epochs=100, imgsz=640, batch=-1, patience=20,
    close_mosaic=10, degrees=10.0, flipud=0.5, fliplr=0.5, scale=0.5,
    save_period=10, seed=42,
    project=RUN_PROJECT, name=RUN_NAME,
)
print('best:', results.save_dir)


In [ ]:
# 5. test split 평가
best = f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt'
m = YOLO(best)
mm = m.val(data=DATA, split='test', imgsz=640, batch=8)
print('BOX  mAP50 %.3f  mAP50-95 %.3f' % (mm.box.map50, mm.box.map))
print('MASK mAP50 %.3f  mAP50-95 %.3f' % (mm.seg.map50, mm.seg.map))
for i,n in mm.names.items():
    if i < len(mm.seg.ap50): print(f'  {n:10s} mask AP50 {mm.seg.ap50[i]:.3f}')


In [ ]:
# 6. 예측 시각화 몇 장 -> Drive 복사
import glob, random, shutil, os
base = os.path.dirname(DATA)
imgs = glob.glob(f'{base}/images/test/*'); random.seed(0); random.shuffle(imgs)
m.predict(imgs[:24], imgsz=640, conf=0.25, save=True, line_width=2, project='/content/pred', name='test')
shutil.make_archive(f'{RUN_PROJECT}/{RUN_NAME}/pred_test', 'zip', '/content/pred/test')
print('saved pred zip')


In [ ]:
# 7. ONNX export (opset=17, imgsz=640) — Orin TensorRT 용
onnx_path = m.export(format='onnx', opset=17, imgsz=640, simplify=True, nms=False)
print('ONNX:', onnx_path)

import shutil
for f in ['best.pt','best.onnx']:
    src = f'{RUN_PROJECT}/{RUN_NAME}/weights/{f}'
    if os.path.exists(src): print('OK  ', src)

# 출력 텐서 형상 확인 (Orin 파서 설계 기준)
import onnx
o = onnx.load(onnx_path)
for out in o.graph.output:
    dims = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(out.name, dims)


## 산출물 (Drive `silorobot_runs/cc_seg_v1/`)
- `weights/best.pt`, `weights/best.onnx`
- `results.png`, `args.yaml`, `pred_test.zip`

## 다음
1. `best.pt` 로컬 다운로드 → `python local_test/run_local_infer.py`
2. `best.onnx` → `python local_test/onnx_parity_check.py`
3. Orin 배포 → `docs/orin_tensorrt_deploy.md`


In [ ]:
# 8. (선택) 전체 실행 완료 후 런타임 자동 해제 — Colab 컴퓨팅 단위 절약
#  - "모두 실행"으로 여기까지 오면 런타임을 반납하고 세션이 끊긴다.
#  - 산출물은 이미 Drive(silorobot_runs/cc_seg_v1/)에 저장돼 있으므로 안전.
#  - 중간 셀에서 에러가 나면 실행이 멈춰서 이 셀은 안 돌아간다(디버깅용으로 런타임 유지됨).
#  - 자동 해제를 원치 않으면 이 셀을 실행하지 말 것.
try:
    from google.colab import runtime
    print("결과는 Drive에 저장됨. 런타임을 해제한다...")
    runtime.unassign()
except Exception as e:
    print("런타임 해제 스킵:", e)
